# Laboratorio L6 - Passphrase e k-anonimato

B.4 - Cybersicurezza e IA

- parte 1: generazione di passphrase con parole casuali
- parte 2: simulazione della verifica di una password con il k-anonimato, come fa il servizio Pwned Passwords di Have I Been Pwned

Non inserire password reali.

## 1. Elenco di parole

L'elenco contiene 256 parole italiane. Con 256 parole ogni parola estratta a caso vale log2(256) = 8 bit di entropia.

L'elenco può essere pubblico: la sicurezza dipende dal numero di combinazioni, non dalla segretezza dell'elenco.

In [ ]:
import math

PAROLE = [
    "abete", "acqua", "agenda", "aglio", "albero", "alce", "alga", "alloro", "altalena", "ambra",
    "amico", "ancora", "anello", "anguria", "antenna", "ape", "arancia", "arco", "argento", "aria",
    "armadio", "arpa", "asino", "astro", "atlante", "autunno", "avena", "azzurro", "baffo", "balena",
    "bambola", "banana", "barca", "basilico", "baule", "becco", "bicchiere", "biglia", "binario", "biscotto",
    "bosco", "bottiglia", "bottone", "bruco", "bussola", "caffe", "calamaio", "calcio", "camino", "campana",
    "candela", "cane", "canguro", "canoa", "cappello", "capra", "carciofo", "carota", "castello", "cavallo",
    "cedro", "cervo", "cesto", "chiave", "ciliegia", "cinghiale", "cipolla", "coccinella", "cometa", "conchiglia",
    "coniglio", "corda", "corvo", "cratere", "cuscino", "dado", "delfino", "diamante", "divano", "drago",
    "duna", "eclisse", "edera", "elica", "elmo", "faggio", "falco", "farfalla", "faro", "fenice",
    "ferro", "fico", "filo", "fiore", "fiume", "foca", "foglia", "fontana", "foresta", "formica",
    "forno", "fragola", "frutto", "fulmine", "fungo", "gabbiano", "galassia", "gatto", "gelato", "geranio",
    "ghiaccio", "ghianda", "ghiro", "giacca", "giglio", "giraffa", "girasole", "gomitolo", "gomma", "gufo",
    "iceberg", "isola", "lago", "lampada", "lampone", "lana", "lanterna", "lavagna", "leone", "lepre",
    "libro", "limone", "lince", "lucertola", "luna", "lupo", "magnete", "mandorla", "mantello", "mappa",
    "marmo", "martello", "matita", "medusa", "mela", "melone", "menta", "mirtillo", "montagna", "mulino",
    "muschio", "nave", "nebbia", "neve", "nido", "noce", "nuvola", "oasi", "oca", "olivo",
    "ombrello", "onda", "orologio", "orso", "ortica", "orzo", "ossigeno", "pacco", "palla", "palma",
    "panca", "panda", "pane", "papavero", "pappagallo", "pastello", "patata", "pavone", "penna", "pepe",
    "pera", "pesca", "pettine", "pianeta", "pietra", "pigna", "pinguino", "pino", "pioggia", "piuma",
    "polpo", "pomodoro", "ponte", "porto", "pozzo", "prato", "prisma", "pulcino", "quaderno", "quarzo",
    "quercia", "radice", "ragno", "rame", "rana", "razzo", "riccio", "riso", "rondine", "rosa",
    "rubino", "ruota", "sabbia", "sale", "salice", "sasso", "scala", "scoiattolo", "sedia", "segale",
    "seme", "sentiero", "serpente", "sfera", "slitta", "sole", "spiga", "spugna", "stella", "stivale",
    "strada", "tamburo", "tappeto", "tartaruga", "tavolo", "tazza", "teiera", "telaio", "tenda", "terra",
    "tigre", "timone", "topo", "torre", "trattore", "treno", "tromba", "tulipano", "uccello", "ulivo",
    "uovo", "uva", "vapore", "vaso", "vela", "ventaglio"
]

print("parole nell'elenco:", len(PAROLE))
print("bit per parola:", math.log2(len(PAROLE)))

## 2. Generatore di passphrase

- `secrets.choice(elenco)`: sceglie un elemento a caso usando un generatore di numeri casuali adatto ai segreti. Il modulo `random`, pensato per simulazioni e giochi, non va usato per password
- `"-".join(lista)`: unisce gli elementi della lista separandoli con `-`

In [ ]:
import secrets

def genera_passphrase(n_parole, separatore="-"):
    return separatore.join(secrets.choice(PAROLE) for _ in range(n_parole))

for n in [4, 6, 8]:
    frase = genera_passphrase(n)
    bit = n * math.log2(len(PAROLE))
    print(f"{n} parole ({bit:.0f} bit, {len(frase)} caratteri): {frase}")

**Domanda (esercizio 2)**: confrontare l'entropia delle passphrase con quella delle password della tabella di L5 (per esempio 8 caratteri di ogni tipo: 52,4 bit). Quante parole servono per superarla?

Risposta:

## 3. Simulazione del k-anonimato

Il servizio Pwned Passwords riceve solo i primi 5 caratteri esadecimali dell'hash SHA-1 della password e risponde con tutti i suffissi noti che iniziano con quel prefisso. Il confronto avviene sul computer dell'utente.

Nella simulazione il "servizio" è un piccolo archivio locale di password fittizie considerate trapelate; per ogni prefisso vengono aggiunti circa 800 suffissi casuali, per rendere realistico il numero di suffissi restituiti. Nessun dato esce dal computer.

- `hashlib.sha1(testo.encode()).hexdigest().upper()`: calcola l'hash SHA-1 e lo scrive in esadecimale maiuscolo
- `hash[:5]` e `hash[5:]`: i primi 5 caratteri e i restanti (slicing)

In [ ]:
import hashlib

def sha1(testo):
    return hashlib.sha1(testo.encode("utf-8")).hexdigest().upper()

import random

# password fittizie considerate trapelate
TRAPELATE = ["password", "123456", "Estate2026!", "juventus2008", "qwertyuiop", "Scu0la2026"]
archivio = {sha1(p) for p in TRAPELATE}

def servizio_range(prefisso, altri=800):
    """Il servizio riceve solo il prefisso e restituisce tutti i suffissi con quel prefisso.

    Il servizio reale contiene centinaia di milioni di hash: per ogni prefisso restituisce
    circa un migliaio di suffissi. Qui quei suffissi vengono simulati con valori casuali
    (sempre gli stessi per lo stesso prefisso), senza occupare memoria.
    """
    veri = [h[5:] for h in archivio if h.startswith(prefisso)]
    rng = random.Random(prefisso)          # simulazione: non serve a generare segreti
    simulati = ["%035X" % rng.getrandbits(140) for _ in range(altri)]
    return veri + simulati

In [ ]:
def verifica(password):
    h = sha1(password)
    prefisso, suffisso = h[:5], h[5:]
    risposta = servizio_range(prefisso)          # unica comunicazione con il servizio
    trovata = suffisso in risposta               # confronto locale
    print(f"password controllata : {password}")
    print(f"hash SHA-1 completo  : {h}")
    print(f"inviato al servizio  : {prefisso}")
    print(f"suffissi ricevuti    : {len(risposta)}")
    print(f"esito                : {'COMPROMESSA' if trovata else 'non trovata'}")
    print()

verifica("Estate2026!")
verifica(genera_passphrase(6))

**Domande (esercizio 3)**

- Che cosa conosce il servizio della password controllata?
- Perché conviene che la risposta contenga molti suffissi e non uno solo?
- Nel servizio reale i suffissi restituiti per ogni prefisso sono diverse centinaia: quanto è "nascosta" la password tra di essi?

Risposte:

## 4. Approfondimento (esercizio 5)

Quante parole servono per superare i 70,5 bit di una password di 15 minuscole casuali, con l'elenco di 256 parole e con un elenco di 7 776 parole?

`math.ceil(x)` arrotonda per eccesso all'intero successivo.

In [ ]:
obiettivo = 15 * math.log2(26)
for w in [256, 7776]:
    print(f"elenco di {w} parole: servono {math.ceil(obiettivo / math.log2(w))} parole")